# Preprocessing Data untuk Association Rule Mining
Dataset: `retail_store_sales.csv`

## 3.3.1. Membaca dan Membersihkan Data

In [ ]:
# Install library yang dibutuhkan (jalankan sekali, lalu RESTART KERNEL)
!pip install mlxtend

In [ ]:
import pandas as pd

# Membaca dataset
df = pd.read_csv('retail_store_sales.csv')

# Menampilkan info awal dataset
print(f"Jumlah baris awal: {len(df)}")
print(f"Kolom: {list(df.columns)}")
print(f"\nJumlah nilai kosong pada kolom 'Item': {df['Item'].isna().sum()}")
df.head()

In [ ]:
# Membuang baris yang tidak memiliki nama 'Item'
# (karena barang yang namanya kosong tidak bisa dianalisis)
df_clean = df.dropna(subset=['Item']).copy()

print(f"Jumlah baris setelah cleaning: {len(df_clean)}")
print(f"Baris yang dihapus: {len(df) - len(df_clean)}")

## 3.3.2. Mengonversi Data ke dalam Format Transaksi

**Pendekatan:** Mengelompokkan data per **Customer ID per Bulan** (bukan per Transaction ID).

**Alasan:**
- Setiap Transaction ID hanya berisi 1 item → tidak bisa menghasilkan association rules
- Grouping per Customer (tanpa batas waktu) → semua customer membeli semua kategori → rules trivial
- Grouping per Customer per Bulan → menghasilkan variasi yang cukup untuk rules bermakna

In [ ]:
# Parse tanggal dan buat kolom Year-Month
df_clean['Transaction Date'] = pd.to_datetime(df_clean['Transaction Date'])
df_clean['YearMonth'] = df_clean['Transaction Date'].dt.to_period('M').astype(str)

# Menyederhanakan nama kategori yang terlalu panjang
df_clean['Category'] = df_clean['Category'].replace({
    'Computers and electric accessories': 'Computers',
    'Electric household essentials': 'Electric'
})

# Buat Basket ID = Customer + Month
df_clean['Basket'] = df_clean['Customer ID'] + '_' + df_clean['YearMonth']

# Mengelompokkan kategori unik per basket
baskets = df_clean.groupby('Basket')['Category'].apply(lambda x: sorted(set(x))).reset_index()

# Menampilkan jumlah basket
print(f"Jumlah basket (customer-bulan): {len(baskets)}")
print(f"Jumlah kategori unik: {df_clean['Category'].nunique()}")

# Menampilkan 5 basket pertama
print("\n5 Basket Pertama:")
for i, row in baskets.head(5).iterrows():
    print(f"  {row['Basket']}: {row['Category']}")

## 3.3.3. Konversi ke One-Hot Encoding (per Kategori)

In [ ]:
# Membuat one-hot encoding dari basket ke kategori
basket_cat = df_clean[['Basket', 'Category']].drop_duplicates()
basket_cat['Value'] = True

df_encoded = basket_cat.pivot_table(
    index='Basket', 
    columns='Category', 
    values='Value', 
    fill_value=False, 
    aggfunc='first'
)
df_encoded = df_encoded.reset_index(drop=True)

# Pastikan tipe data boolean
for col in df_encoded.columns:
    df_encoded[col] = df_encoded[col].astype(bool)

print(f"Dimensi data encoded: {df_encoded.shape}")
print(f"Kolom kategori: {list(df_encoded.columns)}")

# Statistik jumlah kategori per basket
cat_counts = df_encoded.sum(axis=1)
print(f"\nJumlah kategori per basket:")
print(f"  Min: {cat_counts.min()}")
print(f"  Max: {cat_counts.max()}")
print(f"  Rata-rata: {cat_counts.mean():.2f}")
print(f"  Basket dengan < 8 kategori: {(cat_counts < 8).sum()}/{len(df_encoded)} ({(cat_counts < 8).mean()*100:.1f}%)")

In [ ]:
# Menampilkan data yang sudah di-encode
print("Bentuk Data Setelah One-Hot Encoding:")
df_encoded.head(10)

In [ ]:
# Menampilkan support (proporsi) per kategori
print("Support per kategori:")
print(df_encoded.mean().round(4))

## 3.3.4. Menyimpan Data yang Telah Diproses

In [ ]:
# Menyimpan hasil one-hot encoding ke file CSV
file_path = "retail_category_encoded.csv"
df_encoded.to_csv(file_path, index=False)

print(f"File '{file_path}' berhasil disimpan!")
print(f"Dimensi: {df_encoded.shape[0]} baris x {df_encoded.shape[1]} kolom")

# Jika dijalankan di Google Colab, uncomment baris berikut untuk download otomatis:
# from google.colab import files
# files.download(file_path)